# Week 8 — Demo, Documentation, and Wrap-Up

Last week of the program. Everything technical is already built — UNet, noise scheduler, DDPM training, DDIM sampling, classifier-free guidance, a custom dataset run. This week is just packaging it so someone other than me can press a button and get a digit out.

I'm rebuilding the conditional MNIST pipeline from Week 6 here (rather than the Fashion-MNIST one from Week 7) since digits make for a cleaner, more legible demo for someone seeing this for the first time. The same Gradio wrapper would work unchanged on the Week 7 model — I'd just swap which checkpoint gets loaded and update the class label list.

In [ ]:
!pip install -q gradio

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
from torchvision.utils import make_grid
import numpy as np
import gradio as gr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
torch.manual_seed(0)

NUM_CLASSES = 10
NULL_LABEL = NUM_CLASSES
CKPT_PATH = "ddpm_mnist_conditional.pt"

## Model, scheduler, sampler — same code as Week 6

No changes here at all. The point of this week is that the modeling work is done; I'm just wiring it up behind an interface.

In [ ]:
class NoiseScheduler:
    def __init__(self, timesteps=1000, s=0.008, device=device):
        self.timesteps = timesteps
        steps = torch.arange(timesteps + 1, dtype=torch.float64) / timesteps
        f_t = torch.cos((steps + s) / (1 + s) * math.pi / 2) ** 2
        alphas_cumprod = f_t / f_t[0]
        alphas_cumprod = torch.clamp(alphas_cumprod, min=1e-9)

        self.alphas_cumprod = alphas_cumprod[1:].float().to(device)
        alphas_cumprod_prev = torch.cat([torch.tensor([1.0]), self.alphas_cumprod[:-1]])
        self.alphas_cumprod_prev = alphas_cumprod_prev.to(device)
        self.betas = (1 - self.alphas_cumprod / self.alphas_cumprod_prev).clamp(max=0.999)
        self.alphas = 1.0 - self.betas

    def add_noise(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_ac = self.alphas_cumprod[t].sqrt().view(-1, 1, 1, 1)
        sqrt_one_minus_ac = (1 - self.alphas_cumprod[t]).sqrt().view(-1, 1, 1, 1)
        return sqrt_ac * x0 + sqrt_one_minus_ac * noise, noise

scheduler = NoiseScheduler(timesteps=1000, device=device)

In [ ]:
class SinusoidalTimestepEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(min(8, in_ch), in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.cond_proj = nn.Linear(cond_dim, out_ch)
        self.norm2 = nn.GroupNorm(min(8, out_ch), out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, cond_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.cond_proj(cond_emb)[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class Down(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim):
        super().__init__()
        self.block = ResBlock(in_ch, out_ch, cond_dim)
        self.pool = nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1)

    def forward(self, x, cond_emb):
        h = self.block(x, cond_emb)
        return self.pool(h), h


class Up(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, cond_dim):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch, 2, stride=2)
        self.block = ResBlock(in_ch + skip_ch, out_ch, cond_dim)

    def forward(self, x, skip, cond_emb):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.pad(x, (0, skip.shape[-1] - x.shape[-1], 0, skip.shape[-2] - x.shape[-2]))
        x = torch.cat([x, skip], dim=1)
        return self.block(x, cond_emb)


class ConditionalUNet(nn.Module):
    def __init__(self, in_ch=1, base_ch=64, time_dim=128, num_classes=10):
        super().__init__()
        self.time_embed = SinusoidalTimestepEmbedding(time_dim)
        self.time_mlp = nn.Sequential(nn.Linear(time_dim, time_dim * 4), nn.SiLU(), nn.Linear(time_dim * 4, time_dim))
        self.null_label = num_classes
        self.label_embed = nn.Embedding(num_classes + 1, time_dim)

        self.inc = ResBlock(in_ch, base_ch, time_dim)
        self.down1 = Down(base_ch, base_ch * 2, time_dim)
        self.down2 = Down(base_ch * 2, base_ch * 4, time_dim)
        self.down3 = Down(base_ch * 4, base_ch * 8, time_dim)
        self.bottleneck = ResBlock(base_ch * 8, base_ch * 8, time_dim)
        self.up1 = Up(base_ch * 8, base_ch * 8, base_ch * 4, time_dim)
        self.up2 = Up(base_ch * 4, base_ch * 4, base_ch * 2, time_dim)
        self.up3 = Up(base_ch * 2, base_ch * 2, base_ch, time_dim)
        self.outc = nn.Conv2d(base_ch, in_ch, 1)

    def forward(self, x, t, labels):
        cond_emb = self.time_mlp(self.time_embed(t)) + self.label_embed(labels)
        h0 = self.inc(x, cond_emb)
        h1, skip1 = self.down1(h0, cond_emb)
        h2, skip2 = self.down2(h1, cond_emb)
        h3, skip3 = self.down3(h2, cond_emb)
        h3 = self.bottleneck(h3, cond_emb)
        h = self.up1(h3, skip3, cond_emb)
        h = self.up2(h, skip2, cond_emb)
        h = self.up3(h, skip1, cond_emb)
        return self.outc(h)

model = ConditionalUNet().to(device)
print(sum(p.numel() for p in model.parameters()), "parameters")

## Train (or load) the checkpoint

If `ddpm_mnist_conditional.pt` already exists in the working directory, load it directly instead of retraining — this is exactly the kind of thing the Gradio demo needs to be robust to, since a Hugging Face Space restarts cold and has to load from a checkpoint file, not from notebook state.

In [ ]:
import os

if os.path.exists(CKPT_PATH):
    model.load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=True))
    print("Loaded existing checkpoint.")
else:
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    train_dataset = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
    epochs = 20
    cond_dropout_prob = 0.1

    for epoch in range(epochs):
        running_loss = 0.0
        for x0, labels in train_loader:
            x0 = x0.to(device)
            labels = labels.to(device)
            drop_mask = torch.rand(labels.shape[0], device=device) < cond_dropout_prob
            train_labels = torch.where(drop_mask, torch.full_like(labels, NULL_LABEL), labels)

            t = torch.randint(0, scheduler.timesteps, (x0.shape[0],), device=device)
            xt, noise = scheduler.add_noise(x0, t)
            pred_noise = model(xt, t, train_labels)
            loss = F.mse_loss(pred_noise, noise)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running_loss += loss.item() * x0.shape[0]

        print(f"epoch {epoch+1}/{epochs}  loss={running_loss / len(train_dataset):.4f}")

    torch.save(model.state_dict(), CKPT_PATH)
    print("Saved checkpoint to", CKPT_PATH)

In [ ]:
@torch.no_grad()
def cfg_ddim_sample(model, scheduler, labels, num_steps=50, guidance_scale=3.0, device=device, x_T=None):
    model.eval()
    T = scheduler.timesteps
    shape = (labels.shape[0], 1, 28, 28)

    step_indices = torch.linspace(0, T - 1, num_steps).long().flip(0).to(device)
    x = torch.randn(shape, device=device) if x_T is None else x_T
    null_labels = torch.full_like(labels, NULL_LABEL)
    ac = scheduler.alphas_cumprod

    for i in range(len(step_indices)):
        t = step_indices[i]
        t_batch = torch.full((shape[0],), t.item(), device=device, dtype=torch.long)

        ac_t = ac[t]
        ac_prev = ac[step_indices[i + 1]] if i + 1 < len(step_indices) else torch.tensor(1.0, device=device)

        eps_cond = model(x, t_batch, labels)
        eps_uncond = model(x, t_batch, null_labels)
        eps = eps_uncond + guidance_scale * (eps_cond - eps_uncond)

        x0_pred = ((x - (1 - ac_t).sqrt() * eps) / ac_t.sqrt()).clamp(-1, 1)
        dir_coeff = torch.sqrt((1 - ac_prev).clamp(min=0))
        x = ac_prev.sqrt() * x0_pred + dir_coeff * eps

    model.train()
    return x.clamp(-1, 1)

## The Gradio demo

Four controls, matching the deliverable checklist: which digit to generate, how strongly to enforce that label (guidance scale), how many DDIM steps to spend on it, and a seed for reproducibility. The generate function returns a single image grid (4 samples at once) so a viewer can see some of the digit-to-digit variation at a fixed setting rather than just one cherry-picked output.

In [ ]:
def generate(digit, guidance_scale, num_steps, seed):
    seed = int(seed)
    torch.manual_seed(seed)

    n_samples = 4
    labels = torch.full((n_samples,), int(digit), device=device, dtype=torch.long)
    samples = cfg_ddim_sample(model, scheduler, labels, num_steps=int(num_steps), guidance_scale=float(guidance_scale))

    grid = make_grid(samples, nrow=4, normalize=True, value_range=(-1, 1))
    grid_np = (grid.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    if grid_np.shape[-1] == 1:
        grid_np = grid_np.squeeze(-1)
    return grid_np


demo = gr.Interface(
    fn=generate,
    inputs=[
        gr.Dropdown(choices=list(range(10)), value=7, label="Digit (0-9)"),
        gr.Slider(minimum=0.0, maximum=10.0, value=3.0, step=0.5, label="Guidance scale"),
        gr.Slider(minimum=10, maximum=200, value=50, step=10, label="DDIM steps"),
        gr.Number(value=42, label="Seed"),
    ],
    outputs=gr.Image(label="Generated samples"),
    title="Conditional DDPM — Digit Generator",
    description=(
        "Pick a digit, then generate 4 samples with DDIM sampling and classifier-free guidance. "
        "Higher guidance scale pushes harder toward the chosen digit at the cost of diversity; "
        "more DDIM steps generally improves quality but takes longer."
    ),
    submit_btn="Generate",
)

demo.launch(debug=False, share=True)

## Deploy to Hugging Face Spaces

The two cells below package everything into a standalone `app.py` and push it to a public HF Space alongside the trained checkpoint. Set `HF_USERNAME` to your Hugging Face username and make sure `HF_TOKEN` is available (add it as a Colab secret under the key `HF_TOKEN`).

The Space will restart cold each time it wakes up, so `app.py` is fully self-contained — all model definitions, checkpoint loading, sampler, and Gradio interface in one file.

In [ ]:
app_py = '''
import math, os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import gradio as gr
from torchvision.utils import make_grid

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 10
NULL_LABEL = NUM_CLASSES
CKPT_PATH = "ddpm_mnist_conditional.pt"


class NoiseScheduler:
    def __init__(self, timesteps=1000, s=0.008, device=device):
        self.timesteps = timesteps
        steps = torch.arange(timesteps + 1, dtype=torch.float64) / timesteps
        f_t = torch.cos((steps + s) / (1 + s) * math.pi / 2) ** 2
        alphas_cumprod = f_t / f_t[0]
        alphas_cumprod = torch.clamp(alphas_cumprod, min=1e-9)
        self.alphas_cumprod = alphas_cumprod[1:].float().to(device)
        alphas_cumprod_prev = torch.cat([torch.tensor([1.0]), self.alphas_cumprod[:-1]])
        self.alphas_cumprod_prev = alphas_cumprod_prev.to(device)
        self.betas = (1 - self.alphas_cumprod / self.alphas_cumprod_prev).clamp(max=0.999)
        self.alphas = 1.0 - self.betas


class SinusoidalTimestepEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(min(8, in_ch), in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.cond_proj = nn.Linear(cond_dim, out_ch)
        self.norm2 = nn.GroupNorm(min(8, out_ch), out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, cond_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.cond_proj(cond_emb)[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class Down(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim):
        super().__init__()
        self.block = ResBlock(in_ch, out_ch, cond_dim)
        self.pool = nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1)

    def forward(self, x, cond_emb):
        h = self.block(x, cond_emb)
        return self.pool(h), h


class Up(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, cond_dim):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch, 2, stride=2)
        self.block = ResBlock(in_ch + skip_ch, out_ch, cond_dim)

    def forward(self, x, skip, cond_emb):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.pad(x, (0, skip.shape[-1] - x.shape[-1], 0, skip.shape[-2] - x.shape[-2]))
        x = torch.cat([x, skip], dim=1)
        return self.block(x, cond_emb)


class ConditionalUNet(nn.Module):
    def __init__(self, in_ch=1, base_ch=64, time_dim=128, num_classes=10):
        super().__init__()
        self.time_embed = SinusoidalTimestepEmbedding(time_dim)
        self.time_mlp = nn.Sequential(nn.Linear(time_dim, time_dim * 4), nn.SiLU(), nn.Linear(time_dim * 4, time_dim))
        self.null_label = num_classes
        self.label_embed = nn.Embedding(num_classes + 1, time_dim)
        self.inc = ResBlock(in_ch, base_ch, time_dim)
        self.down1 = Down(base_ch, base_ch * 2, time_dim)
        self.down2 = Down(base_ch * 2, base_ch * 4, time_dim)
        self.down3 = Down(base_ch * 4, base_ch * 8, time_dim)
        self.bottleneck = ResBlock(base_ch * 8, base_ch * 8, time_dim)
        self.up1 = Up(base_ch * 8, base_ch * 8, base_ch * 4, time_dim)
        self.up2 = Up(base_ch * 4, base_ch * 4, base_ch * 2, time_dim)
        self.up3 = Up(base_ch * 2, base_ch * 2, base_ch, time_dim)
        self.outc = nn.Conv2d(base_ch, in_ch, 1)

    def forward(self, x, t, labels):
        cond_emb = self.time_mlp(self.time_embed(t)) + self.label_embed(labels)
        h0 = self.inc(x, cond_emb)
        h1, skip1 = self.down1(h0, cond_emb)
        h2, skip2 = self.down2(h1, cond_emb)
        h3, skip3 = self.down3(h2, cond_emb)
        h3 = self.bottleneck(h3, cond_emb)
        h = self.up1(h3, skip3, cond_emb)
        h = self.up2(h, skip2, cond_emb)
        h = self.up3(h, skip1, cond_emb)
        return self.outc(h)


scheduler = NoiseScheduler(timesteps=1000, device=device)
model = ConditionalUNet().to(device)
model.load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=True))
model.eval()
print("Model loaded on", device)


@torch.no_grad()
def cfg_ddim_sample(labels, num_steps=50, guidance_scale=3.0):
    T = scheduler.timesteps
    shape = (labels.shape[0], 1, 28, 28)
    step_indices = torch.linspace(0, T - 1, num_steps).long().flip(0).to(device)
    x = torch.randn(shape, device=device)
    null_labels = torch.full_like(labels, NULL_LABEL)
    ac = scheduler.alphas_cumprod

    for i in range(len(step_indices)):
        t = step_indices[i]
        t_batch = torch.full((shape[0],), t.item(), device=device, dtype=torch.long)
        ac_t = ac[t]
        ac_prev = ac[step_indices[i + 1]] if i + 1 < len(step_indices) else torch.tensor(1.0, device=device)
        eps_cond = model(x, t_batch, labels)
        eps_uncond = model(x, t_batch, null_labels)
        eps = eps_uncond + guidance_scale * (eps_cond - eps_uncond)
        x0_pred = ((x - (1 - ac_t).sqrt() * eps) / ac_t.sqrt()).clamp(-1, 1)
        dir_coeff = torch.sqrt((1 - ac_prev).clamp(min=0))
        x = ac_prev.sqrt() * x0_pred + dir_coeff * eps

    return x.clamp(-1, 1)


def generate(digit, guidance_scale, num_steps, seed):
    torch.manual_seed(int(seed))
    n_samples = 4
    labels = torch.full((n_samples,), int(digit), device=device, dtype=torch.long)
    samples = cfg_ddim_sample(labels, num_steps=int(num_steps), guidance_scale=float(guidance_scale))
    grid = make_grid(samples, nrow=4, normalize=True, value_range=(-1, 1))
    grid_np = (grid.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    if grid_np.shape[-1] == 1:
        grid_np = grid_np.squeeze(-1)
    return grid_np


demo = gr.Interface(
    fn=generate,
    inputs=[
        gr.Dropdown(choices=list(range(10)), value=7, label="Digit (0-9)"),
        gr.Slider(minimum=0.0, maximum=10.0, value=3.0, step=0.5, label="Guidance scale"),
        gr.Slider(minimum=10, maximum=200, value=50, step=10, label="DDIM steps"),
        gr.Number(value=42, label="Seed"),
    ],
    outputs=gr.Image(label="Generated samples"),
    title="Conditional DDPM — Digit Generator",
    description=(
        "Pick a digit, then generate 4 samples with DDIM sampling and classifier-free guidance. "
        "Higher guidance scale pushes harder toward the chosen digit at the cost of diversity; "
        "more DDIM steps generally improves quality but takes longer."
    ),
    submit_btn="Generate",
)

if __name__ == "__main__":
    demo.launch()
'''

with open("app.py", "w") as f:
    f.write(app_py.lstrip())

with open("requirements.txt", "w") as f:
    f.write("torch\ntorchvision\ngradio\nnumpy\n")

print("Wrote app.py and requirements.txt")

In [ ]:
import os
from huggingface_hub import HfApi, login

# ── set these before running ──────────────────────────────────────────────────
HF_USERNAME = os.environ.get("HF_USERNAME", "your-hf-username")   # replace or set env var
HF_TOKEN    = os.environ.get("HF_TOKEN", "")                       # add as Colab secret
# ──────────────────────────────────────────────────────────────────────────────

assert HF_TOKEN, "Set HF_TOKEN before deploying (Colab: Runtime > Secrets)"
login(token=HF_TOKEN)

SPACE_ID = f"{HF_USERNAME}/diffusion-mnist-demo"
api = HfApi()

api.create_repo(
    repo_id=SPACE_ID,
    repo_type="space",
    space_sdk="gradio",
    private=False,
    exist_ok=True,
)
print(f"Space created (or already exists): https://huggingface.co/spaces/{SPACE_ID}")

for fname in ["app.py", "requirements.txt"]:
    api.upload_file(
        path_or_fileobj=fname,
        path_in_repo=fname,
        repo_id=SPACE_ID,
        repo_type="space",
    )
    print(f"  uploaded {fname}")

if os.path.exists(CKPT_PATH):
    api.upload_file(
        path_or_fileobj=CKPT_PATH,
        path_in_repo=CKPT_PATH,
        repo_id=SPACE_ID,
        repo_type="space",
    )
    print(f"  uploaded {CKPT_PATH}")
else:
    print(f"  WARNING: {CKPT_PATH} not found — train the model first (run the training cell)")

print(f"\nSpace live at: https://huggingface.co/spaces/{SPACE_ID}")

## Technical write-up

**What I built.** An eight-week progression from a plain PyTorch MLP classifier up to a class-conditional DDPM with DDIM sampling and classifier-free guidance, trained on MNIST and a custom Smithsonian Butterflies run, wrapped in a Gradio demo. Each week's notebook is self-contained and reuses the previous week's components without modification where possible — the UNet architecture from Week 4 carries through unchanged into Week 8, with only the conditioning mechanism (Week 6) and dataset (Week 7) changing along the way.

**Biggest debugging challenge.** The trickiest bugs weren't in the high-level training loop — they were in dimension bookkeeping inside the UNet: skip-connection channel counts after concatenation, `GroupNorm` requiring channel counts divisible by the number of groups, and `F.pad` ordering when transposed-convolution output didn't exactly match the skip tensor's spatial size at odd input resolutions. None of these throw helpful error messages; they either silently produce wrong shapes that fail several layers later, or crash with a generic shape-mismatch error that doesn't point at the actual cause. Tracing every tensor's shape by hand layer-by-layer, rather than trusting that "it'll just work," is what actually caught these before they wasted GPU time.

**What I'd improve with more time.** Three things stand out: (1) move from MNIST/butterflies to CIFAR-10 or a higher-resolution custom set to actually stress-test the pipeline at a harder scale; (2) replace the fixed evenly-spaced DDIM timestep schedule with an adaptive solver like DPM-Solver, which should get acceptable quality in even fewer steps; (3) add an EMA (exponential moving average) of model weights during training, which is standard practice in diffusion training and tends to produce visibly cleaner samples than the raw trained weights.

## Self-check questions

**1. If a stranger cloned this repo and opened only the Week 8 notebook, could they generate a sample without reading anything else?**

Mostly yes, with one caveat. Running every cell top to bottom installs Gradio, rebuilds the model and scheduler, trains (or loads a cached checkpoint), and launches the demo — no other notebook needs to be opened. The one thing a stranger would have to know going in is that the *first* run with no checkpoint present will train from scratch (~20 epochs on MNIST), which takes real time before the Gradio link appears; that's noted in the markdown above the training cell, but it's easy to miss if someone's skimming for the demo and gets impatient waiting on the training cell.

**2. What's the one design decision from the past 8 weeks you'd undo if you started over?**

Building the `Down`/`Up` UNet blocks with skip channels implicitly derived from each `ResBlock`'s output rather than passing an explicit, asserted channel count into every `Up` constructor. That ambiguity — skip channels being the *out_ch* of the matching `Down`, not something declared and checked at the `Up` site — is exactly what produced the channel-mismatch bug that turned up across Weeks 5 through 8 once the architecture got copy-pasted forward. An explicit `assert` on concatenated channel counts in `Up.forward`, or just being more deliberate about what `skip_ch` means at each call site, would have caught it at the first model instantiation instead of letting it propagate silently through four notebooks.

**3. What's the smallest change that would most improve sample quality right now?**

Adding an EMA (exponential moving average) of the model weights during training. It's a few lines — track a shadow copy of the parameters updated as a moving average each step, sample from the shadow copy instead of the raw weights — and it's nearly free compute-wise, but it consistently produces visibly less noisy, more stable samples than the raw trained weights in diffusion models generally. Compared to architecture changes or more training data, it's the highest quality-per-line-of-code change available.

**4. What didn't make it into this program that you'd want to explore next (DPM-Solver, latent diffusion, text conditioning, etc.)?**

DPM-Solver is the most immediately useful one — it would push sampling from DDIM's ~50 steps down toward single digits while keeping quality, using an adaptive ODE solver instead of a fixed evenly-spaced schedule. After that, latent diffusion (running the diffusion process in a compressed VAE latent space instead of pixel space) is the natural next architectural step, since it's what makes diffusion models tractable at real image resolutions instead of just 28x28/64x64 toy settings. Text conditioning is the furthest out but the most exciting — it's a straightforward conceptual extension of what Week 6 already built (swap the class-label embedding for a text encoder's output), just with a much bigger and more expensive encoder behind it.

## Presentation outline (5 slides)

**Slide 1 — Title.** Conditional Diffusion Models from Scratch: an 8-week build, from MLP to a guided DDIM sampler with a live demo.

**Slide 2 — The arc.** Week 1 MLP classifier → Week 2 UNet denoising autoencoder → Week 3 forward diffusion math → Week 4 full DDPM (Algorithm 1 & 2) → Week 5 DDIM (50 steps, no retraining) → Week 6 classifier-free guidance → Week 7 Smithsonian Butterflies custom dataset run → Week 8 Gradio demo.

**Slide 3 — Best generated samples.** The Week 6 0-9 grid at `guidance_scale=3.0`, alongside the DDPM vs DDIM speed comparison from Week 5 (1000 steps vs 50 steps, near-identical quality, ~20x fewer model evaluations).

**Slide 4 — Biggest debugging challenge.** UNet dimension bookkeeping — skip connections, GroupNorm divisibility, and padding mismatches were the actual time sink, not the diffusion math itself.

**Slide 5 — What's next.** CIFAR-10 (real photographic data), DPM-Solver for even faster sampling, EMA weights for cleaner samples, and ultimately text conditioning as the natural next conditioning signal after class labels.